<a href="https://colab.research.google.com/github/elandler/repo-pruebas/blob/main/DS2_EL_Discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREGUNTA:
¿Qué variables asociadas al nivel educativo y sector económico influyen en la empleabilidad de personas mayores de 45 años en municipios españoles, y qué sectores presentan mayor capacidad de absorción laboral para este grupo etario?

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3 as sql
from pathlib import Path

In [51]:
# Ejemplo 1 con un CSV público en sepe.es
url = "https://sede.sepe.gob.es/es/portaltrabaja/resources/sede/datos_abiertos/datos/Dtes_empleo_por_municipios_2025_csv.csv"

# Leer el CSV sin encabezado inicialmente para que la primera y segunda fila sean parte de los datos
df1_temp = pd.read_csv(url, encoding='latin1', sep=';', header=None, low_memory=False)

# La segunda fila (índice 1) contiene los nombres de las columnas deseados
new_columns = df1_temp.iloc[1].tolist()

# Asignar estos nombres como las columnas del DataFrame
df1_temp.columns = new_columns

# Eliminar las dos primeras filas (el identificador y la fila que ahora es el encabezado) y resetear el índice
df1 = df1_temp[2:].reset_index(drop=True)

# Mostrar primeras filas
df1.head()

,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,total Dtes Empleo,Dtes Empleo hombre edad < 25,Dtes Empleo hombre edad 25 -45,Dtes Empleo hombre edad >=45,Dtes Empleo mujer edad < 25,Dtes Empleo mujer edad 25 -45,Dtes Empleo mujer edad >=45,Dtes EmpleoAgricultura,Dtes Empleo Industria,Dtes Empleo Construcción,Dtes Empleo Servicios,Dtes Empleo Sin empleo Anterior
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,148,8,24,44,<5,22,48,39,<5,8,91,6
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,148,8,13,39,5,22,61,53,5,8,72,10
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,2642,121,432,601,106,588,794,643,77,200,1536,186
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,34,<5,<5,14,<5,5,10,7,<5,<5,18,<5
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,73,<5,7,23,<5,15,24,19,13,5,35,<5


In [52]:
# Ejemplo 2 con un CSV público en sepe.es
url = "https://sede.sepe.gob.es/es/portaltrabaja/resources/sede/datos_abiertos/datos/Contratos_por_municipios_2025_csv.csv"

# Leer el CSV sin encabezado inicialmente para que la primera y segunda fila sean parte de los datos
df2_temp = pd.read_csv(url, encoding='latin1', sep=';', header=None, low_memory=False)

# La segunda fila (índice 1) contiene los nombres de las columnas deseados
new_columns_df2 = df2_temp.iloc[1].tolist()

# Asignar estos nombres como las columnas del DataFrame
df2_temp.columns = new_columns_df2

# Eliminar las dos primeras filas (el identificador y la fila que ahora es el encabezado) y resetear el índice
df2 = df2_temp[2:].reset_index(drop=True)

# Mostrar primeras filas
df2.head()

,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,Total Contratos,Contratos iniciales indefinidos hombres,Contratos iniciales temporales hombres,Contratos convertidos en indefinidos hombres,Contratos iniciales indefinidos mujeres,Contratos iniciales temporales mujeres,Contratos convertidos en indefinidos mujeres,Contratos Agricultura,Contratos Industria,Contratos Construcción,Contratos Servicios
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,46,8,7,<5,12,18,0,9,0,<5,36
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,29,14,7,0,0,8,0,12,<5,<5,13
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,411,168,46,20,123,50,<5,175,9,61,166
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,6,0,<5,0,<5,<5,0,<5,0,0,<5
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,11,<5,<5,0,0,8,0,<5,<5,<5,5


In [53]:
# Asignar df1 a un nombre más descriptivo
df_demandantes_empleo = df1

# Asignar df2 a un nombre más descriptivo
df_contratos_registrados = df2

print("DataFrames renombrados exitosamente:")
print(f"df_demandantes_empleo: {df_demandantes_empleo.shape[0]} filas, {df_demandantes_empleo.shape[1]} columnas")
print(f"df_contratos_registrados: {df_contratos_registrados.shape[0]} filas, {df_contratos_registrados.shape[1]} columnas")


DataFrames renombrados exitosamente:
df_demandantes_empleo: 97620 filas, 20 columnas
df_contratos_registrados: 97620 filas, 19 columnas


In [54]:
df_demandantes_empleo.head()


,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,total Dtes Empleo,Dtes Empleo hombre edad < 25,Dtes Empleo hombre edad 25 -45,Dtes Empleo hombre edad >=45,Dtes Empleo mujer edad < 25,Dtes Empleo mujer edad 25 -45,Dtes Empleo mujer edad >=45,Dtes EmpleoAgricultura,Dtes Empleo Industria,Dtes Empleo Construcción,Dtes Empleo Servicios,Dtes Empleo Sin empleo Anterior
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,148,8,24,44,<5,22,48,39,<5,8,91,6
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,148,8,13,39,5,22,61,53,5,8,72,10
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,2642,121,432,601,106,588,794,643,77,200,1536,186
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,34,<5,<5,14,<5,5,10,7,<5,<5,18,<5
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,73,<5,7,23,<5,15,24,19,13,5,35,<5


In [55]:
df_contratos_registrados.head()


,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,Total Contratos,Contratos iniciales indefinidos hombres,Contratos iniciales temporales hombres,Contratos convertidos en indefinidos hombres,Contratos iniciales indefinidos mujeres,Contratos iniciales temporales mujeres,Contratos convertidos en indefinidos mujeres,Contratos Agricultura,Contratos Industria,Contratos Construcción,Contratos Servicios
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,46,8,7,<5,12,18,0,9,0,<5,36
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,29,14,7,0,0,8,0,12,<5,<5,13
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,411,168,46,20,123,50,<5,175,9,61,166
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,6,0,<5,0,<5,<5,0,<5,0,0,<5
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,11,<5,<5,0,0,8,0,<5,<5,<5,5


In [56]:
print (df_demandantes_empleo.shape)
print (df_contratos_registrados.shape)

(97620, 20)
(97620, 19)


In [57]:
def clean_small_values(x):
    if isinstance(x, str) and "<" in x:
        return 2
    # Asegurarse de que el valor sea un string antes de intentar convertirlo
    try:
        return int(x)
    except ValueError:
        return x # Devolver el valor original si no se puede convertir a int

# Primero, limpiar los nombres de las columnas para eliminar espacios en blanco al inicio o al final
# y remover el prefijo '1 ' si existe en los nombres de las columnas, y normalizar espacios internos.
df_demandantes_empleo.columns = [" ".join(col.replace('1 ', '').strip().split()) for col in df_demandantes_empleo.columns.astype(str)]
df_contratos_registrados.columns = [" ".join(col.replace('1 ', '').strip().split()) for col in df_contratos_registrados.columns.astype(str)]

# Lista de columnas numéricas que necesitan limpieza en df_demandantes_empleo
numeric_cols_demandantes = [
    'Código mes',
    'total Dtes Empleo',
    'Dtes Empleo hombre edad < 25',
    'Dtes Empleo hombre edad 25 -45',
    'Dtes Empleo hombre edad >=45',
    'Dtes Empleo mujer edad < 25',
    'Dtes Empleo mujer edad 25 -45',
    'Dtes Empleo mujer edad >=45',
    'Dtes EmpleoAgricultura',
    'Dtes Empleo Industria',
    'Dtes Empleo Construcción',
    'Dtes Empleo Servicios',
    'Dtes Empleo Sin empleo Anterior'
]

# Lista de columnas numéricas que necesitan limpieza en df_contratos_registrados
numeric_cols_contratos = [
    'Código mes',
    'Total Contratos',
    'Contratos iniciales indefinidos hombres',
    'Contratos iniciales temporales hombres',
    'Contratos convertidos en indefinidos hombres',
    'Contratos iniciales indefinidos mujeres',
    'Contratos iniciales temporales mujeres',
    'Contratos convertidos en indefinidos mujeres',
    'Contratos Agricultura',
    'Contratos Industria',
    'Contratos Construcción',
    'Contratos Servicios'
]

# Aplicar la función de limpieza solo a las columnas numéricas seleccionadas
for col in numeric_cols_demandantes:
    if col in df_demandantes_empleo.columns:
        df_demandantes_empleo[col] = df_demandantes_empleo[col].apply(lambda x: clean_small_values(x) if isinstance(x, str) else x)

for col in numeric_cols_contratos:
    if col in df_contratos_registrados.columns:
        df_contratos_registrados[col] = df_contratos_registrados[col].apply(lambda x: clean_small_values(x) if isinstance(x, str) else x)

In [58]:
df_demandantes_empleo.head()

,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,total Dtes Empleo,Dtes Empleo hombre edad < 25,Dtes Empleo hombre edad 25 -45,Dtes Empleo hombre edad >=45,Dtes Empleo mujer edad < 25,Dtes Empleo mujer edad 25 -45,Dtes Empleo mujer edad >=45,Dtes EmpleoAgricultura,Dtes Empleo Industria,Dtes Empleo Construcción,Dtes Empleo Servicios,Dtes Empleo Sin empleo Anterior
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,148,8,24,44,2,22,48,39,2,8,91,6
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,148,8,13,39,5,22,61,53,5,8,72,10
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,2642,121,432,601,106,588,794,643,77,200,1536,186
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,34,2,2,14,2,5,10,7,2,2,18,2
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,73,2,7,23,2,15,24,19,13,5,35,2


In [59]:
df_contratos_registrados.head()

,Código mes,mes,Código de CA,Comunidad Autónoma,Codigo Provincia,Provincia,Codigo Municipio,Municipio,Total Contratos,Contratos iniciales indefinidos hombres,Contratos iniciales temporales hombres,Contratos convertidos en indefinidos hombres,Contratos iniciales indefinidos mujeres,Contratos iniciales temporales mujeres,Contratos convertidos en indefinidos mujeres,Contratos Agricultura,Contratos Industria,Contratos Construcción,Contratos Servicios
0,202501,Enero de 2025,1,Andalucía,4,Almería,04001,Abla,46,8,7,2,12,18,0,9,0,2,36
1,202501,Enero de 2025,1,Andalucía,4,Almería,04002,Abrucena,29,14,7,0,0,8,0,12,2,2,13
2,202501,Enero de 2025,1,Andalucía,4,Almería,04003,Adra,411,168,46,20,123,50,2,175,9,61,166
3,202501,Enero de 2025,1,Andalucía,4,Almería,04004,Albánchez,6,0,2,0,2,2,0,2,0,0,2
4,202501,Enero de 2025,1,Andalucía,4,Almería,04005,Alboloduy,11,2,2,0,0,8,0,2,2,2,5
